# C0 · Qwen3.8-27B 비전 인코더 (ViT + M-RoPE) 순전파 검증

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_llm_dit/blob/main/notebooks/verify/C0_qwen38_vision.ipynb)

원본: transformers `qwen3_5` 의 `Qwen3_5VisionModel` + `Qwen3_5Model.get_rope_index`
(코드 읽은 기준: tag `v5.20.0` = commit `48e4bcd`, `src/transformers/models/qwen3_5/modeling_qwen3_5.py` + `src/transformers/vision_utils.py`).
Qwen3.8-27B 체크포인트(`Qwen/Qwen3.8-27B`)의 `model_type` 이 `qwen3_5` 라 이 코드를 그대로 씁니다.
내 구현: `miniarch/models/qwen38_vision.py` (텍스트 쪽은 A6a 의 `miniarch/models/qwen36.py` 재사용).

부품
- patch embed: 프레임 2장 × 16×16 픽셀 patch 를 Conv3d(커널 = stride) 한 번으로 벡터 하나로. patch 순서는 2×2 merge 블록 순서
- 절대 위치: 학습된 48×48 위치 표를 이미지 격자 크기로 bilinear 보간 (align_corners=True) 해서 더함
- 2D RoPE: head_dim 을 [행 주파수 | 열 주파수] × 2 로 채워 전체 회전
- attention: 프레임 하나 안에서만 양방향 full attention (Qwen2.5-VL 의 window attention 은 없음)
- patch merger: LayerNorm → 이웃 2×2 patch 이어붙이기 → MLP → LM 폭 (27B: 5120)
- LM 쪽 M-RoPE: 이미지 토큰은 (t, h, w) 3축 위치, 다음 텍스트는 max(h, w) 만큼만 건너뜀. 회전 채널은 interleaved 배치

축소 config (모든 경로가 한 번씩 쓰이도록 고름)
- ViT 2층, 폭 64, head 4 (head_dim 16 → 축마다 주파수 2개), patch 4, 위치 표 4×4
- 입력: 이미지 (1, 4, 6) + 2프레임 비디오 (2, 6, 4) + 아주 작은 이미지 (1, 2, 2) 를 한 열로 패킹
  → 위치 표보다 큰/작은 격자 보간, 끝 clamp, 프레임별 attention 구간, 비디오 프레임 반복 위치가 모두 지나감
- 최종 비교는 merger 출력 (LM 에 들어가는 토큰), 층별 출력도 비교합니다.
- 자체 점검: get_rope_index 와 위치 일치 (padding·비디오 프레임 분할 포함), M-RoPE cos/sin 일치,
  ViT + Qwen36 을 이어 붙인 미니 VLM 이 원본 `Qwen3_5Model` 과 같은지.

순서: 0. 환경 준비 → 1. 원본 구조 (HF 헤더) → 2. 원본 모델 동작 확인 → 3. 내 구현 == 원본 (층별 출력 비교) → 4. 단계 전용 자체 점검 → 5. 결과 저장

**런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 위에서부터 실행하세요. 다른 모델 노트북과 독립적으로 돌아갑니다.

## 0. 환경 준비

In [ ]:
import os
REPO_DIR = "/content/tmp_llm_dit"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/HisameOgasahara/tmp_llm_dit.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
if not os.environ.get("MINIARCH_SKIP_PIP"):   # 전체 실행 노트북이 한 번만 설치하고 넘길 때 건너뜀
    !pip -q install -U -r requirements-colab.txt
    # Colab 기본 torchao 가 최신 diffusers 와 버전이 안 맞아 import 가 깨짐 → 쓰지 않으므로 제거
    !pip -q uninstall -y torchao

In [ ]:
import sys, torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU로 바꾸세요)")
!git log -1 --format='repo commit: %h %s'

In [ ]:
from miniarch import verify as V
from miniarch.verify import stages
S = stages.get("C0")
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:   # gated 모델 헤더/원격 코드용. Colab 왼쪽 🔑(보안 비밀)에 HF_TOKEN 등록.
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

CHECKS = []   # (이름, 통과, 오차) → 마지막에 저장
def report(name, ok, err=float("nan")):
    CHECKS.append({"check": name, "ok": bool(ok), "max_abs": float(err)})
    print(f"{'✅' if ok else '❌'} {name:<44} max_abs={err:.2e}")

## 원본 구조 (HF 헤더)

실제 체크포인트의 safetensors 헤더만 읽어 가중치 이름, shape, dtype을 봅니다 (다운로드 없음).
**구현 검증이 아니라 원본 구조 확인용**입니다. routed expert는 개수가 많아 숨깁니다.

In [ ]:
def peek(repo, pattern, subfolder="", limit=60):
    print(f"\n===== {repo}  /{pattern}/")
    try:
        cfg = V.fetch_config(repo, subfolder)
        cfg = cfg.get("text_config", cfg)
        keys = ["model_type", "hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim"]
        print({k: cfg[k] for k in keys if k in cfg})
        h = V.read_model_header(repo, subfolder=subfolder, pattern=pattern)
        V.show({k: v for k, v in h.items() if ".experts." not in k}, limit)
    except Exception as e:
        print(f"⚠️ {type(e).__name__}: {e}")

peek('Qwen/Qwen3.8-27B', 'visual\\.(patch_embed|pos_embed|blocks\\.0\\.|merger)', subfolder='')

## 원본 모델 동작 확인

축소 config로 원본을 만들고 한 번 forward 합니다. 여기서 실패하면 라이브러리 버전 문제입니다 (내 구현과 무관).

In [ ]:
V.set_deterministic(0)
ref, inputs = S.ref()
ref = ref.to(dev).eval()
with torch.no_grad():
    out = ref(**{k: v.to(dev) if torch.is_tensor(v) else v for k, v in inputs.items()})
t = out[0] if isinstance(out, tuple) else getattr(out, "last_hidden_state", None)
t = t if t is not None else out.sample
print(f"{type(ref).__name__}  params={sum(p.numel() for p in ref.parameters()):,}  out={tuple(t.shape)}")

## 내 구현 == 원본

원본 가중치에 잡음을 넣고(초기값 0/1 때문에 복사 누락이 가려지지 않도록) 내 구현에 복사한 뒤, 층별 출력과 최종 출력을 비교합니다.

In [ ]:
row = V.cases.run_case(S.CASE)
print(row["status"]); print(row["detail"])
report("원본 동치 (" + S.CASE.reference + ")", row["status"].startswith("✅"))

## 단계 전용 자체 점검

### M-RoPE 위치 == 원본 get_rope_index

텍스트·이미지·비디오가 섞인 배치 (왼쪽 padding 포함) 에서 3축 위치와 delta 가 원본과 정확히 같은지 봅니다.
- 비디오 (2, 4, 4) 는 프레임 사이에 타임스탬프 텍스트가 끼므로 프레임마다 따로 위치를 받습니다.
- 원본 함수는 모델 메서드라 config 만 흉내 낸 객체로 호출합니다 (가중치 불필요).

In [ ]:
import types
from transformers.models.qwen3_5.modeling_qwen3_5 import Qwen3_5Model
from miniarch.models import qwen38_vision as Q
ns = types.SimpleNamespace(config=types.SimpleNamespace(vision_config=types.SimpleNamespace(spatial_merge_size=2)))
ns.get_vision_position_ids = types.MethodType(Qwen3_5Model.get_vision_position_ids, ns)

# (종류, 토큰 수): 0 텍스트, 1 이미지, 2 비디오 프레임
rows = [[(0, 2), (1, 6), (0, 3), (2, 4), (0, 1), (2, 4), (0, 2)],
        [(0, 4), (1, 1), (0, 2), (1, 6), (0, 5), (2, 2), (0, 1)]]
img_thw = torch.tensor([[1, 4, 6], [1, 2, 2], [1, 6, 4]])
vid_thw = torch.tensor([[2, 4, 4], [1, 2, 4]])
T_ = max(sum(n for _, n in r) for r in rows) + 3
mm = torch.zeros(len(rows), T_, dtype=torch.int32)
am = torch.zeros(len(rows), T_, dtype=torch.long)
for b, r in enumerate(rows):
    types_ = sum(([k] * n for k, n in r), [])
    mm[b, T_ - len(types_):] = torch.tensor(types_)
    am[b, T_ - len(types_):] = 1                                     # 왼쪽 padding
ids = torch.zeros(len(rows), T_, dtype=torch.long)
p_ref, d_ref = Qwen3_5Model.get_rope_index(ns, ids, mm, image_grid_thw=img_thw.clone(),
                                           video_grid_thw=vid_thw.clone(), attention_mask=am)
p_me, d_me = Q.rope_index(mm, img_thw.clone(), vid_thw.clone(), attention_mask=am, merge=2)
print("t/h/w 위치 (0번 열):\n", p_me[:, 0])
print("delta:", d_me.flatten().tolist())
err = (p_ref - p_me).abs().max().item()
report("M-RoPE 3축 위치 == get_rope_index", p_ref.shape == p_me.shape and err == 0, err)
report("rope delta 일치", torch.equal(d_ref.cpu().long(), d_me.cpu().long()), 0.0)
MROPE_POS = p_me

### M-RoPE cos/sin == 원본 Qwen3_5TextRotaryEmbedding

위 위치로 회전 각을 만들 때, 주파수 i 를 i mod 3 = 0/1/2 → t/h/w 축으로 고르는 interleaved 배치가 원본과 같은지 봅니다.
(rotary 폭 16 → 주파수 8개, mrope_section [3, 3, 2] 로 세 축이 모두 쓰이게 함)

In [ ]:
from transformers.models.qwen3_5.configuration_qwen3_5 import Qwen3_5TextConfig
from transformers.models.qwen3_5.modeling_qwen3_5 import Qwen3_5TextRotaryEmbedding
tc = Qwen3_5TextConfig(hidden_size=64, num_attention_heads=4, num_key_value_heads=2, head_dim=64,
                       rope_parameters={"rope_type": "default", "rope_theta": 10000.0,
                                        "partial_rotary_factor": 0.25, "mrope_section": [3, 3, 2]})
rot = Qwen3_5TextRotaryEmbedding(tc)
cos_r, sin_r = rot(torch.zeros(1), MROPE_POS)
cos_m, sin_m = Q.mrope_cos_sin(MROPE_POS, rot.inv_freq, [3, 3, 2])
err = max((cos_r - cos_m).abs().max().item(), (sin_r - sin_m).abs().max().item())
report("M-RoPE cos/sin (interleaved) 일치", cos_r.shape == cos_m.shape and err < 1e-6, err)

### 미니 VLM == 원본 Qwen3_5Model

내 ViT + `qwen36.py` 텍스트 층 + M-RoPE 를 이어 붙인 `Qwen38VL` 이 원본 `Qwen3_5Model` (vision + 하이브리드 LM) 과
같은 last_hidden_state 를 내는지 봅니다. 이미지 2장 (격자 4×4, 2×8) + 2프레임 비디오 2개, 시퀀스 26.
가중치 이름이 같아 `load_state_dict` 로 그대로 옮깁니다.

In [ ]:
import dataclasses
V.set_deterministic(0)
vlm, vin = S.ref_vlm()
V.perturb_params(vlm)
vlm = vlm.to(dev).float().eval()
for mod in vlm.modules():
    try:
        mod.config._attn_implementation = "eager"
    except Exception:
        pass
mine = Q.Qwen38VL(Q.vl_config_from_hf(vlm.config)).to(dev).eval()
res = mine.load_state_dict(vlm.state_dict(), strict=False)
print("missing:", res.missing_keys[:5], " unexpected:", res.unexpected_keys[:5])
report("VLM state_dict 키 일치", not res.missing_keys and not res.unexpected_keys, 0.0)
vin = {k: v.to(dev) for k, v in vin.items()}
with torch.no_grad():
    a = vlm(**vin).last_hidden_state
    b = mine(**vin)
err = (a - b).abs().max().item()
report("미니 VLM == Qwen3_5Model", a.shape == b.shape and err < 1e-4, err)

mine2 = Q.Qwen38VL(dataclasses.replace(mine.config, use_mrope=False)).to(dev).eval()
mine2.load_state_dict(mine.state_dict())
with torch.no_grad():
    d = (mine2(**vin) - b).abs().max().item()
report("M-RoPE 끄면 (1D 위치) 출력이 바뀜", d > 1e-4, d)

### 프레임별 attention · ablation 스위치

- 비디오 2번째 프레임 픽셀만 바꾸면 1번째 프레임과 다른 이미지의 ViT 출력 (merger 전) 은 그대로여야 합니다.
- 2D RoPE / 보간 위치 임베딩을 끄거나 attention 범위를 비디오 전체(`clip`)로 넓히면 출력이 바뀌는지 봅니다.

In [ ]:
import dataclasses
ref, inputs = S.ref()
V.perturb_params(ref)
m = Q.verify_pair(ref)["mine"]
m.load_state_dict(ref.state_dict())
m = m.to(dev).eval()
inp = {k: v.to(dev) for k, v in inputs.items()}
with torch.no_grad():
    base = m(**inp)
    x2 = inp["hidden_states"].clone()
    x2[48:72] += 1.0                                                  # 비디오 (2, 6, 4) 의 2번째 프레임 = patch 48..71
    alt = m(hidden_states=x2, grid_thw=inp["grid_thw"]).last_hidden_state
keep = torch.cat([torch.arange(0, 48), torch.arange(72, alt.shape[0])]).to(dev)
err = (alt[keep] - base.last_hidden_state[keep]).abs().max().item()
report("다른 프레임/이미지 출력 불변", err < 1e-5, err)
d = (alt[48:72] - base.last_hidden_state[48:72]).abs().max().item()
report("바꾼 프레임 출력은 변함", d > 1e-4, d)
for kw in (dict(use_rope=False), dict(use_abs_pos=False), dict(attn_scope="clip")):
    m2 = Q.Qwen38Vision(dataclasses.replace(m.config, **kw)).to(dev).eval()
    m2.load_state_dict(m.state_dict())
    with torch.no_grad():
        d = (m2(**inp).pooler_output - base.pooler_output).abs().max().item()
    report(f"ablation {kw} → 출력 변화", d > 1e-4, d)

## 결과 저장

- `results/`에 점검 결과를 저장합니다.
- 노트북 출력까지 남기려면 **파일 → GitHub에 사본 저장** → 이 저장소의 `notebooks/verify/C0_qwen38_vision.ipynb`.

In [ ]:
import datetime, pandas as pd
os.makedirs("results", exist_ok=True)
df = pd.DataFrame(CHECKS)
p = f"results/verify_C0_{datetime.datetime.now():%Y%m%d-%H%M}.csv"
df.to_csv(p, index=False)
print(df.to_string(index=False)); print("saved:", p)
print("전체:", "✅ 모두 통과" if df.ok.all() else "❌ 실패 있음")